In [1]:
import torch, tqdm, time, os, json
import numpy as np
import pandas as pd
import pytorch_lightning as pl
from models.est_model import GroundPressureModel
from torchmetrics import Accuracy
from torch.utils.data import DataLoader, Dataset, random_split
from pytorch_lightning.callbacks import DeviceStatsMonitor, EarlyStopping
from sklearn.metrics import r2_score

In [2]:
monitor = DeviceStatsMonitor()

In [3]:
# set train setting
save_model = True
train_num_epoch = 50000
min_loss = 100
dl_workers = 0

In [4]:
gpu = torch.device('cuda')
torch.set_float32_matmul_precision('high')

In [5]:
# load hyperparameter of json file
with open('.' + os.sep + os.path.join('models', 'params_dnn_20220207-012403.json'), 'r') as file:
    hyper_params = json.load(file)

n_inputs = hyper_params['n_of_inputs']
n_outputs = hyper_params['n_of_outputs']
n_layers = hyper_params['n_of_layers']

In [6]:
model = GroundPressureModel(n_inputs, n_layers, n_outputs)
print(model)

GroundPressureModel(
  (loss): MSELoss()
  (model): Sequential(
    (0): Linear(in_features=4, out_features=24, bias=True)
    (1): BasicBlock(
      (basic_block): Sequential(
        (0): Linear(in_features=24, out_features=24, bias=True)
        (1): Linear(in_features=24, out_features=24, bias=True)
      )
      (shortcut_block): Sequential()
      (relu): ReLU()
    )
    (2): BasicBlock(
      (basic_block): Sequential(
        (0): Linear(in_features=24, out_features=24, bias=True)
        (1): Linear(in_features=24, out_features=24, bias=True)
      )
      (shortcut_block): Sequential()
      (relu): ReLU()
    )
    (3): BasicBlock(
      (basic_block): Sequential(
        (0): Linear(in_features=24, out_features=24, bias=True)
        (1): Linear(in_features=24, out_features=24, bias=True)
      )
      (shortcut_block): Sequential()
      (relu): ReLU()
    )
    (4): BasicBlock(
      (basic_block): Sequential(
        (0): Linear(in_features=24, out_features=24, bias=Tru

In [7]:
data = pd.read_csv('./resources/sim_data.csv')
feature_names = ['lift_weight(ton)', 'lift_height(m)', 'rising_angle(deg)', 'swing_angle(deg)']
target_names = ['left_ground_pressure_min(kg/cm2)', 'left_ground_pressure_max(kg/cm2)', 'left_pressure_length(m)', 'right_ground_pressure_min(kg/cm2)', 'right_ground_pressure_max(kg/cm2)', 'right_pressure_length(m)']

feature_data = []
target_data = []

for feature_name in feature_names:
    feature_data.append(data[feature_name])

for target_name in target_names:
    target_data.append(data[target_name])

feature_data = np.array(feature_data, dtype=np.float32).T
target_data = np.array(target_data, dtype=np.float32).T

class TensorDataset(Dataset):
    def __init__(self, feature, target):
        self.x_data = torch.tensor(feature)
        self.y_data = torch.tensor(target)
        self.len = self.x_data.shape[0]

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.len

train_dataset = TensorDataset(feature_data, target_data)
train_data_loader = DataLoader(train_dataset, batch_size=len(train_dataset), num_workers=16)

In [8]:
# train model
early_stop_callback = EarlyStopping(monitor='train_loss', mode='min', verbose=True, min_delta=0.001, patience=50)
trainer = pl.Trainer(callbacks=[early_stop_callback], accelerator='cuda', enable_progress_bar=True, max_epochs=10000)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [9]:
trainer.fit(model=model, train_dataloaders=train_data_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type       | Params
-------------------------------------
0 | loss  | MSELoss    | 0     
1 | model | Sequential | 1.5 K 
2 | relu  | ReLU       | 0     
-------------------------------------
1.5 K     Trainable params
0         Non-trainable params
1.5 K     Total params
0.006     Total estimated model params size (MB)
/home/jinbeom/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3.8/site-packages/pytorch_lightning/loops/fit_loop.py:280: PossibleUserWarning: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
  rank_zero_warn(


Epoch 0: 100%|██████████| 1/1 [00:00<00:00,  1.27it/s, v_num=53]

Metric train_loss improved. New best score: 977.627


Epoch 1: 100%|██████████| 1/1 [00:00<00:00,  1.69it/s, v_num=53]

Metric train_loss improved by 488.796 >= min_delta = 0.001. New best score: 488.831


...

Epoch 121: 100%|██████████| 1/1 [00:00<00:00,  1.68it/s, v_num=53]

Metric train_loss improved by 0.005 >= min_delta = 0.001. New best score: 0.290


Epoch 171: 100%|██████████| 1/1 [00:00<00:00,  1.65it/s, v_num=53]

Monitored metric train_loss did not improve in the last 50 records. Best score: 0.290. Signaling Trainer to stop.


Epoch 171: 100%|██████████| 1/1 [00:00<00:00,  1.62it/s, v_num=53]


In [11]:
torch.save(model.state_dict(), './models/est_ground_pressure.pt')

In [10]:
model.eval()
for data in train_dataset:
    print(r2_score(data[1].detach().numpy() ,model(data[0]).detach().numpy()))
    print(model(data[0]).detach().numpy())
    print(data[1].detach().numpy())

0.9625717833688605
[4.1512012  0.02284412 3.1185653  3.9657648  0.17763354 3.1601121 ]
[3.97 0.   3.72 3.97 0.   3.72]
0.9360484304375025
[2.495569   0.02649607 3.9868991  3.421681   0.15694368 4.0013247 ]
[2.61 0.   4.1  4.58 0.   4.1 ]
0.9568211371750365
[1.6764454  0.00968565 5.0291905  3.299996   0.12305237 5.0934434 ]
[1.15 0.   5.59 4.13 0.   5.59]
0.9408632402521132
[3.3538988  0.02770473 4.772085   3.9906251  0.07214756 4.6752315 ]
[2.93 0.   5.04 2.93 0.   5.04]
0.9778631015325118
[2.1097758  0.00737132 4.7848353  3.4479303  0.13412203 4.7823987 ]
[2.16 0.   5.34 3.36 0.   5.34]
0.9832605661124462
[ 1.3703195e+00 -4.3391436e-03  5.8462205e+00  3.4323497e+00
  8.4901936e-03  6.0181413e+00]
[1.25 0.   6.53 3.27 0.   6.53]
0.9165023986108907
[ 2.5914938   0.0284995   6.258638    3.9906309  -0.03232939  6.0697265 ]
[2.21 0.   6.68 2.21 0.   6.68]
0.9147179976075271
[ 1.7390095e+00 -2.1107048e-03  5.5144014e+00  3.4855518e+00
  6.4761087e-02  5.5497479e+00]
[1.83 0.04 6.75 2.46 0.0